---
numbering: false
---

# 2.4: Lines and planes in ℝ³

In [1]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys

_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root) not in sys.path:
    sys.path.insert(0, str(_notes_root))
from plot_style import style_plotly

BLUE, ORANGE, PINK = '#3d81f6', 'orange', '#d81a60'


def base2(xrange=(-5, 6), yrange=(-5, 5), width=700, height=580, tick=1):
    fig = style_plotly(go.Figure(), renderer='png')
    fig.update_layout(width=width, height=height, showlegend=False,
                      font=dict(size=17), margin=dict(l=55, r=30, t=30, b=50))
    fig.update_xaxes(title='x', range=xrange, dtick=tick, zeroline=True, zerolinewidth=1.5,
                     constrain='domain', scaleanchor='y', scaleratio=1)
    fig.update_yaxes(title='y', range=yrange, dtick=tick, zeroline=True, zerolinewidth=1.5,
                     constrain='domain')
    return fig


def line2(fig, start, end, color=BLUE, dash='solid', width=2):
    fig.add_trace(go.Scatter(x=[start[0], end[0]], y=[start[1], end[1]], mode='lines',
                            line=dict(color=color, width=width, dash=dash),
                            hoverinfo='skip', showlegend=False))


def label2(fig, xy, label, dx=0, dy=0):
    fig.add_annotation(x=float(xy[0])+dx, y=float(xy[1])+dy, text=label, showarrow=False,
                       font=dict(color='black', size=17))


def point2(fig, xy, label=None, dx=0.35, dy=0.35):
    fig.add_trace(go.Scatter(x=[xy[0]], y=[xy[1]], mode='markers',
                            marker=dict(color='black', size=7), hoverinfo='skip'))
    if label: label2(fig, xy, label, dx, dy)


def vec2(fig, end, label=None, color=BLUE, start=(0, 0), dx=0.25, dy=0.3, width=3):
    fig.add_annotation(x=float(end[0]), y=float(end[1]), ax=float(start[0]), ay=float(start[1]),
                       xref='x', yref='y', axref='x', ayref='y', showarrow=True,
                       text='', arrowhead=3, arrowsize=1.25, arrowwidth=width, arrowcolor=color)
    if label: label2(fig,end,label,dx,dy)


def square2(fig, corner, along, perp, size=0.3):
    corner=np.asarray(corner,dtype=float)
    u=np.asarray(along,dtype=float); u=u/np.linalg.norm(u)*size
    v=np.asarray(perp,dtype=float); v=v/np.linalg.norm(v)*size
    line2(fig,corner+u,corner+u+v,'gray',width=1.5)
    line2(fig,corner+v,corner+u+v,'gray',width=1.5)


def base3():
    fig=style_plotly(go.Figure(),renderer='plotly_mimetype')
    axis=dict(range=[-6,7],dtick=2,showbackground=True,showspikes=False)
    fig.update_layout(width=760,height=620,showlegend=False,font=dict(size=16),
                      margin=dict(l=0,r=0,t=15,b=0),
                      scene=dict(xaxis=dict(title='x',**axis),yaxis=dict(title='y',**axis),
                                 zaxis=dict(title='z',**axis),aspectmode='cube',
                                 camera=dict(eye=dict(x=1.6,y=-2.1,z=1.3))))
    for direction in np.eye(3):
        line3(fig,-6*direction,7*direction,'#9ca3af',width=2)
    return fig


def line3(fig,start,end,color=BLUE,width=5,dash='solid'):
    fig.add_trace(go.Scatter3d(x=[start[0],end[0]],y=[start[1],end[1]],z=[start[2],end[2]],
                              mode='lines',line=dict(color=color,width=width,dash=dash),
                              hoverinfo='skip',showlegend=False))


def vec3(fig,end,label,color=BLUE,start=(0,0,0),offset=(0.25,0.25,0.35)):
    start,end=np.asarray(start,float),np.asarray(end,float)
    direction=(end-start)/np.linalg.norm(end-start)
    line3(fig,start,end-0.15*direction,color,width=7)
    fig.add_trace(go.Cone(x=[end[0]],y=[end[1]],z=[end[2]],u=[direction[0]],v=[direction[1]],w=[direction[2]],
                         anchor='tip',sizemode='absolute',sizeref=0.45,colorscale=[[0,color],[1,color]],
                         showscale=False,hoverinfo='skip'))
    pos=end+offset
    fig.add_trace(go.Scatter3d(x=[pos[0]],y=[pos[1]],z=[pos[2]],mode='text',text=[label],
                              textfont=dict(color='black',size=16),hoverinfo='skip'))


def plane3(fig,normal,color=BLUE,opacity=0.24):
    # Solve n_x*x + n_y*y + n_z*z = 0 for y on a rectangular x,z mesh.
    x,z=np.meshgrid(np.linspace(-5,6,2),np.linspace(-5,6,2))
    n=np.asarray(normal,float)
    y=-(n[0]*x+n[2]*z)/n[1]
    fig.add_trace(go.Surface(x=x.tolist(),y=y.tolist(),z=z.tolist(),
                            colorscale=[[0,color],[1,color]],opacity=opacity,
                            showscale=False,hoverinfo='skip'))
    corners=np.array([[x[0,0],y[0,0],z[0,0]],[x[0,1],y[0,1],z[0,1]],
                      [x[1,1],y[1,1],z[1,1]],[x[1,0],y[1,0],z[1,0]]])
    for i in range(4): line3(fig,corners[i],corners[(i+1)%4],color,width=2)


We have described lines in $\mathbb R^2$ using scalar multiples, normal vectors, and translations. Now we'll carry those ideas into $\mathbb R^3$, where there are both lines and planes.

As in [Chapter 2.1](02-01.ipynb), we'll begin with objects through the origin and then translate them. The vector operations from [Chapter 1.4](../ch01/01-04.ipynb) and the orthogonality test from [Chapter 1.6](../ch01/01-06.ipynb) work in three dimensions too.

**Drag the 3D figures to view the lines and planes from different angles.**

## Lines through the origin

As in the case of $\mathbb{R}^2$, to describe a line through the origin, we
just need to give any nonzero vector along the line.

For example, let

$$
\vec v=
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}.
$$

The line through the origin in the direction of $\vec v$ is

$$
\operatorname{span}(\vec v)
=
\left\{
t
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}
:
t\in\mathbb{R}
\right\}.
$$

In [2]:
fig=base3()
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,BLUE,width=4)
vec3(fig,v,'v',BLUE)
fig.show()

The line through the origin spanned by
$\vec v=\begin{bmatrix}3\\4\\5\end{bmatrix}$.

## Planes as spans

In [Chapter 1.4](../ch01/01-04.ipynb), we formed **linear combinations** by scaling and adding vectors. Here, we use all possible linear combinations of two vectors to fill a plane.

Informally, a plane has two independent directions. To describe a plane through the origin, we need two vectors in the plane that are not scalar multiples of each other. For two vectors, this is what we mean by **independent**.

For example, let

$$
\vec v_1=
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix},
\qquad
\vec v_2=
\begin{bmatrix}
1\\
1\\
1
\end{bmatrix}.
$$

These vectors are not multiples of each other.

In [3]:
fig=base3()
plane3(fig,(1,-2,1))
vec3(fig,(3,4,5),'v<sub>1</sub>',BLUE)
vec3(fig,(1,1,1),'v<sub>2</sub>',ORANGE,offset=(-0.8,0.2,0.4))
fig.show()

The vectors $\vec v_1$ and $\vec v_2$ determine a plane $P$ through
the origin.

The reason that two vectors are enough to describe a plane is that we can take
linear combinations of them to fill out the entire plane.

Let $\ell_1$ be the line through $\vec v_1$, and let $\ell_2$ be the line
through $\vec v_2$.

Consider a vector $\vec w$ in the plane. Complete a parallelogram whose sides
are parallel to $\ell_1$ and $\ell_2$. We can then write

$$
\vec w=\vec w_1+\vec w_2,
$$

where $\vec w_1$ lies on $\ell_1$ and $\vec w_2$ lies on $\ell_2$.

This implies that

$$
\vec w_1=a\vec v_1
$$

for some scalar $a$, and

$$
\vec w_2=b\vec v_2
$$

for some scalar $b$.

Therefore,

$$
\boxed{\vec w=a\vec v_1+b\vec v_2}.
$$

We say that $\vec w$ is a linear combination of $\vec v_1$ and $\vec v_2$.

The following figure gives an exact example. Here

$$
\vec w_1=\vec v_1,
\qquad
\vec w_2=-\vec v_2,
$$

so

$$
\vec w=\vec v_1-\vec v_2
=
\begin{bmatrix}
2\\
3\\
4
\end{bmatrix}.
$$

In [4]:
fig=base3()
plane3(fig,(1,-2,1),opacity=0.15)
v1=np.array([3,4,5]); v2=np.array([1,1,1]); w=v1-v2
line3(fig,-1.1*v1,1.3*v1,BLUE,width=3)
line3(fig,-4*v2,5*v2,ORANGE,width=3)
line3(fig,v1,w,'gray',width=4,dash='dash'); line3(fig,-v2,w,'gray',width=4,dash='dash')
vec3(fig,v1,'w<sub>1</sub> = v<sub>1</sub>',BLUE,offset=(0.4,0.4,0.5))
vec3(fig,-v2,'w<sub>2</sub> = −v<sub>2</sub>',ORANGE,offset=(-0.8,-0.6,-0.5))
vec3(fig,w,'w = v<sub>1</sub> − v<sub>2</sub>',PINK,offset=(-0.5,-0.6,0.5))
fig.show()

The exact parallelogram relation
$\vec w=\vec w_1+\vec w_2=\vec v_1-\vec v_2$.

:::{note} Definition: Span of two vectors

The span of the two vectors $\vec v_1$ and $\vec v_2$ is the set of all linear
combinations

$$
\operatorname{span}(\vec v_1,\vec v_2)
=
\left\{
a\vec v_1+b\vec v_2
:
a,b\in\mathbb{R}
\right\}.
$$

:::

Later, we will generalize this to the span of any number of vectors in
$\mathbb{R}^n$.

Returning to our specific example,

$$
\vec v_1=
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix},
\qquad
\vec v_2=
\begin{bmatrix}
1\\
1\\
1
\end{bmatrix},
$$

the plane $P$ is described by

$$
P
=
\left\{
a\vec v_1+b\vec v_2
:
a,b\in\mathbb{R}
\right\}.
$$

Equivalently,

$$
P
=
\left\{
a
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}
+
b
\begin{bmatrix}
1\\
1\\
1
\end{bmatrix}
:
a,b\in\mathbb{R}
\right\}.
$$

Thus,

$$
P
=
\left\{
\begin{bmatrix}
3a+b\\
4a+b\\
5a+b
\end{bmatrix}
:
a,b\in\mathbb{R}
\right\}.
$$

::::{tip} Activity 1

Use this last description to write out some other vectors in the plane.

:::{tip} Solution
:class: dropdown

We can take any values of $a$ and $b$.

For example, if $a=1$ and $b=-1$, then

$$
\begin{bmatrix}
3a+b\\
4a+b\\
5a+b
\end{bmatrix}
=
\begin{bmatrix}
2\\
3\\
4
\end{bmatrix}
=
\vec v_3.
$$

If $a=2$ and $b=3$, then

$$
\begin{bmatrix}
3a+b\\
4a+b\\
5a+b
\end{bmatrix}
=
\begin{bmatrix}
9\\
11\\
13
\end{bmatrix}
=
\vec v_4.
$$

Notice that taking $a=1$ and $b=0$ recovers $\vec v_1$, while taking $a=0$
and $b=1$ recovers $\vec v_2$.
:::
::::

## Different vectors spanning the same plane

There is nothing special about the original choice $(\vec v_1,\vec v_2)$.
As far as the plane $P$ is concerned, any choice of two independent vectors is
enough to describe the plane.

For example,

$$
P=\operatorname{span}(\vec v_3,\vec v_4).
$$

Why is this?

The key point is that $\vec v_1$ and $\vec v_2$ are themselves linear
combinations of $\vec v_3$ and $\vec v_4$.

Since

$$
\vec v_3=\vec v_1-\vec v_2
$$

and

$$
\vec v_4=2\vec v_1+3\vec v_2,
$$

we have

$$
\vec v_1
=
\frac{3\vec v_3+\vec v_4}{5}
=
\frac35\vec v_3+\frac15\vec v_4
$$

and

$$
\vec v_2
=
\frac{\vec v_4-2\vec v_3}{5}
=
-\frac25\vec v_3+\frac15\vec v_4.
$$

Therefore,

$$
\vec v_1,\vec v_2\in\operatorname{span}(\vec v_3,\vec v_4).
$$

The span of any number of vectors is closed under taking further linear
combinations. One might summarize this with the slogan:

A linear combination of linear combinations is a linear combination.

Indeed, if

$$
\vec v_1=\alpha\vec v_3+\beta\vec v_4
$$

and

$$
\vec v_2=\gamma\vec v_3+\delta\vec v_4,
$$

then

$$
\begin{aligned}
a\vec v_1+b\vec v_2
&=
a(\alpha\vec v_3+\beta\vec v_4)
+
b(\gamma\vec v_3+\delta\vec v_4)\\
&=
(a\alpha+b\gamma)\vec v_3
+
(a\beta+b\delta)\vec v_4\\
&\in
\operatorname{span}(\vec v_3,\vec v_4).
\end{aligned}
$$

This means that any linear combination of $(\vec v_1,\vec v_2)$ is also a
linear combination of $(\vec v_3,\vec v_4)$.

Likewise, any linear combination of $(\vec v_3,\vec v_4)$ is a linear
combination of $(\vec v_1,\vec v_2)$.

Therefore,

$$
\operatorname{span}(\vec v_1,\vec v_2)
=
\operatorname{span}(\vec v_3,\vec v_4).
$$

**Conclusion.**

A plane in $\mathbb{R}^3$ containing the origin may be described as the span
of two independent vectors. This description is not unique, since there are
many different choices of pairs of independent vectors.

## Normal vectors and plane equations

In [Chapter 2.1](02-01.ipynb), the coefficients of a line equation in $\mathbb R^2$ gave us a normal vector. The same connection between equations and orthogonality describes **planes** in $\mathbb R^3$.

It is simplest to start with planes through the origin.

As an example, consider the plane $P$ discussed above:

$$
P=\operatorname{span}(\vec v_1,\vec v_2),
$$

where

$$
\vec v_1=
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix},
\qquad
\vec v_2=
\begin{bmatrix}
1\\
1\\
1
\end{bmatrix}.
$$

We can guess an equation defining this plane by noticing that, for both
$\vec v_1$ and $\vec v_2$,

$$
\text{$x$-coordinate}+\text{$z$-coordinate}
=
2\cdot\text{$y$-coordinate}.
$$

Indeed,

$$
3+5=2(4)
$$

and

$$
1+1=2(1).
$$

This means that $\vec v_1$ and $\vec v_2$ satisfy the equation

$$
x+z=2y,
$$

or equivalently,

$$
x-2y+z=0.
$$

::::{tip} Activity 2

Graph the equation

$$
x-2y+z=0.
$$

Is it the same as the plane $P$?

:::{tip} Solution
:class: dropdown

Yes. Rearranging gives $z=2y-x$. Every point on this graph can be written as

$$
\begin{bmatrix}x\\y\\2y-x\end{bmatrix}
=(y-x)\begin{bmatrix}3\\4\\5\end{bmatrix}
+(4x-3y)\begin{bmatrix}1\\1\\1\end{bmatrix},
$$

so it belongs to $P$. Conversely, substituting $x=3a+b$, $y=4a+b$, and $z=5a+b$ into $x-2y+z$ gives zero. Thus the graph and the span describe exactly the same plane.
:::
::::

What is happening here?

Let

$$
\vec w=
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}.
$$

In [5]:
fig=base3()
plane3(fig,(1,-2,1))
n=np.array([1,-2,1])
line3(fig,-2*n,2*n,PINK,width=3)
vec3(fig,(3,4,5),'v<sub>1</sub>',BLUE)
vec3(fig,(1,1,1),'v<sub>2</sub>',ORANGE,offset=(-0.6,0.2,0.4))
vec3(fig,n,'w',PINK,offset=(0.5,-0.5,0.4))
fig.update_layout(scene_camera=dict(eye=dict(x=2,y=-1.1,z=1.1)))
fig.show()

The vector
$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}$ is perpendicular to the plane
$P$.

The vector

$$
\vec w=
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}
$$

spans the unique line through the origin that is perpendicular to the plane
$P$.

Notice that

$$
\vec v_1\cdot\vec w
=
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}
\cdot
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}
=
3-8+5
=
0,
$$

and

$$
\vec v_2\cdot\vec w
=
\begin{bmatrix}
1\\
1\\
1
\end{bmatrix}
\cdot
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}
=
1-2+1
=
0.
$$

Using the dot product properties from [Chapter 1.6](../ch01/01-06.ipynb), for any linear combination of $\vec v_1$ and $\vec v_2$,

$$
\begin{aligned}
(a\vec v_1+b\vec v_2)\cdot\vec w
&=
a(\vec v_1\cdot\vec w)
+
b(\vec v_2\cdot\vec w)\\
&=
0+0\\
&=
0.
\end{aligned}
$$

**Conclusion.**

The equation

$$
x-2y+z=0
$$

exactly describes the vectors perpendicular to

$$
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}.
$$

This is another way of describing the plane $P$.

**Remark.**

Instead of

$$
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix},
$$

we could take any nonzero scalar multiple of it. For example,

$$
\begin{bmatrix}
3\\
-6\\
3
\end{bmatrix}.
$$

Thus,

$$
3x-6y+3z=0
$$

is another equation describing $P$.

::::{tip} Activity 3

Which method of describing $P$ is easier:

- as the span of two vectors, or
- as the solution set of a single linear equation?

:::{tip} Solution
:class: dropdown

It depends on the task. A span description makes it easy to generate points in the plane by choosing the two coefficients. An equation makes it easy to check whether a given point lies in the plane by substitution. Both describe the same set.
:::
::::

## Lines as intersections of planes

Now let us try to describe lines using equations.

One linear equation in $\mathbb R^3$ with a nonzero normal vector defines a plane. To define a line, we need two equations whose normal vectors are not scalar multiples of each other.

For example, consider the line

$$
\ell
=
\operatorname{span}
\left(
\begin{bmatrix}
3\\
4\\
5
\end{bmatrix}
\right).
$$

One equation satisfied by every vector on this line is

$$
x-2y+z=0.
$$

Can we find another equation that is independent of the first equation?

There are many ways to do this. For example, each of the following equations
is satisfied by every vector on $\ell$:

$$
4x-3y=0,
$$

$$
5y-4z=0,
$$

$$
5x-3z=0,
$$

or

$$
-5x+5y-z=0.
$$

Let

$$
\vec w=
\begin{bmatrix}
1\\
-2\\
1
\end{bmatrix}
$$

and, just for the sake of an example, choose

$$
\vec w'=
\begin{bmatrix}
-5\\
5\\
-1
\end{bmatrix}.
$$

The plane perpendicular to $\vec w$ is

$$
P=\left\{\vec x\in\mathbb{R}^3:\vec x\cdot\vec w=0\right\},
$$

and the plane perpendicular to $\vec w'$ is

$$
P'=\left\{\vec x\in\mathbb{R}^3:\vec x\cdot\vec w'=0\right\}.
$$

Their intersection is the line $\ell$.

In [6]:
fig=base3()
plane3(fig,(1,-2,1),BLUE,0.28)
plane3(fig,(-5,5,-1),ORANGE,0.28)
v=np.array([3,4,5])
line3(fig,-1.1*v,1.3*v,PINK,width=8)
vec3(fig,v,'ℓ',PINK,offset=(0.6,0.3,0.5))
vec3(fig,(1,-2,1),'w',BLUE,offset=(0.2,-0.3,0.4))
vec3(fig,(-5,5,-1),"w′",ORANGE,offset=(0.2,0.2,0.5))
fig.show()

The line
$\ell=\operatorname{span}\left(\begin{bmatrix}3\\4\\5\end{bmatrix}\right)$
is the intersection of two planes through the origin.

Thus, the equations

$$
\begin{cases}
x-2y+z=0,\\
-5x+5y-z=0
\end{cases}
$$

describe the line $\ell$.

**Remark.**

Each equation defines a plane. What we are doing is writing $\ell$ as the
intersection of two planes:

$$
\ell
=
\left\{
\vec x\in\mathbb{R}^3:
\vec x\cdot\vec w=0
\text{ and }
\vec x\cdot\vec w'=0
\right\}.
$$

There are many different ways to do this. Each such way corresponds to choosing
a pair of independent vectors $(\vec w_1,\vec w_2)$ in the plane perpendicular
to $\ell$.

## Affine lines and planes

Affine lines and planes are obtained by translating a line or plane through the
origin by a fixed vector.

An affine line is given by

$$
\vec v_0+\operatorname{span}(\vec v_1)
$$

for some vectors $\vec v_0$ and $\vec v_1$, with $\vec v_1\ne\vec 0$.

An affine plane is given by

$$
\vec v_0+\operatorname{span}(\vec v_1,\vec v_2)
$$

for two independent vectors $\vec v_1$ and $\vec v_2$.

In terms of equations, an affine plane is given by a single linear equation

$$
ax+by+cz=d,
$$

where $\begin{bmatrix}a\\b\\c\end{bmatrix}\ne\begin{bmatrix}0\\0\\0\end{bmatrix}$. When $d=0$, it passes through the origin; when $d\ne0$, the equation is nonhomogeneous and the plane does not contain the origin.

An affine line in $\mathbb{R}^3$ is given by two independent linear equations,

$$
\begin{cases}
a_1x+b_1y+c_1z=d_1,\\
a_2x+b_2y+c_2z=d_2,
\end{cases}
$$

whose corresponding normal vectors

$$
\begin{bmatrix}
a_1\\
b_1\\
c_1
\end{bmatrix},
\qquad
\begin{bmatrix}
a_2\\
b_2\\
c_2
\end{bmatrix}
$$

are independent, meaning that they are not scalar multiples of each other.